# Plasmid size and copy number: an explorer

Companion to [`RESULTS.md`](RESULTS.md) and [`REPORT.md`](REPORT.md).

Two datasets, both fetched and checked against a recorded SHA-256:

| | records | what it is |
|---|---:|---|
| **Ramiro-Martínez et al. 2025** | 3,961 | the curated *modelling* subset, PTU-labelled, no accessions |
| **Maddamsetti et al. 2025** | 12,006 plasmids | Supplementary Data 1, accession-keyed, includes copy numbers below 1 |

Panel B below is a **reproduction** of Maddamsetti's published figure on their own data, and it lands on their published numbers. Panel A imitates the form of the Shahzadi 2026 figure; Shahzadi reuses Maddamsetti's plasmids and copy-number estimates, so the two papers are not independent samples.

**Copy number in both files is modelled, not counted.** Neither ships an uncertainty. Nothing here should be read as resolving a difference smaller than an interval we cannot see.


In [ ]:
# @title Fetch both datasets, and stop if either has moved
import hashlib, io, math, urllib.request
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize

#: Set True only to explore an input that no longer matches the recorded
#: bytes. Every number quoted in this notebook then refers to a file you
#: are not looking at, so it is off by default and the fetch stops.
EXPLORATORY = False

def fetch(url, expected_sha, label):
    req = urllib.request.Request(url, headers={"User-Agent": "plasmid-ecoevo-repro/1.0"})
    raw = urllib.request.urlopen(req, timeout=300).read()
    got = hashlib.sha256(raw).hexdigest()
    print(f"{label}: {len(raw):,} bytes")
    print(f"  sha256 {got}")
    if got == expected_sha:
        print("  matches the recorded input")
    elif EXPLORATORY:
        print("  DIFFERS -- continuing because EXPLORATORY is True.")
        print("  Every number in the markdown refers to the other file.")
    else:
        raise SystemExit(
            f"{label}: input differs from the recorded bytes.\n"
            f"  expected {expected_sha}\n  got      {got}\n"
            "Set EXPLORATORY = True to look at it anyway.")
    return raw

RAMIRO_URL = ("https://raw.githubusercontent.com/PaulaRamiro/NpAUREO/main/Model/curateddataset.csv")
RAMIRO_SHA = "23bee155b1c946bd565840c4cc0f5ac2e1fec6e4192578ba9f6752ee5f58f78f"
MADDAM_URL = ("https://static-content.springer.com/esm/art%3A10.1038%2Fs41467-025-61205-2/MediaObjects/41467_2025_61205_MOESM3_ESM.csv")
MADDAM_SHA = "131faa58f5528d814877cca9ffa8c3e56b41e92518385d740c1b7e530f2bc3a7"

ram_raw = fetch(RAMIRO_URL, RAMIRO_SHA, "Ramiro curated subset")


In [ ]:
# @title Load Ramiro, and name the units trap
df = pd.read_csv(io.BytesIO(ram_raw), sep="\t", encoding="utf-8-sig")
df = df[(df.PCN > 0) & (df.Length_chr > 0)].copy()
df["plasmid_bp"] = 10 ** df.Logsize
df["rel_pct"] = 100 * df.plasmid_bp / df.Length_chr
df["x"] = np.log10(df.rel_pct)     # log10 PERCENT of chromosome
df["y"] = np.log10(df.PCN)

# UNITS. receipt.json stores breakpoint_log10 as log10 of a FRACTION
# (-1.7626). This notebook works in log10 of a PERCENT, so the same knee
# is log10(1.7273) = +0.2373. Mixing the two tests a knee a hundred
# times too small and silently returns a null; it has happened. The
# percentage prints below are the other half of the same trap: where x
# is already log10(percent), 10**c IS the percentage and multiplying
# by 100 again reports a bend at 184% of the chromosome.
KNEE_PCT = math.log10(1.7273321739634590)

print(f"{len(df):,} plasmids, {df.Predicted_ptu.nunique()} PTUs, "
      f"{df.Genus.nunique()} genera")
print(f"PCN {df.PCN.min():.3f} to {df.PCN.max():.2f}")
print()
print("Curation the authors document and the file does not show:")
print("  unassigned PTUs removed -- every record carries a named PTU")
print("  PCN outliers above 100 removed -- max here is 99.50")
print("  PCN below 1 removed -- min here is exactly 1.000")
print("The last is a one-sided truncation of the fitted quantity.")


## 1. The fitting machinery

`dAIC = AIC(segmented) - AIC(single line)`, so **negative favours the bend**. The single line costs two parameters, the hinge four when its knee is estimated and three when the knee is held fixed.


In [ ]:
# @title Fits and model comparison
def ols(x, y):
    k, b = np.polyfit(x, y, 1)
    return k, b, float(((y - (k * x + b)) ** 2).sum())

def hinge(x, y, c):
    M = np.column_stack([np.ones_like(x), x, np.maximum(x - c, 0.0)])
    beta, *_ = np.linalg.lstsq(M, y, rcond=None)
    return float(((y - M @ beta) ** 2).sum()), beta

def best_hinge(x, y, n_grid=400):   # n_grid matches reproduce.py
    grid = np.linspace(np.quantile(x, 0.05), np.quantile(x, 0.95), n_grid)
    sse, c = min((hinge(x, y, g)[0], g) for g in grid)
    _, beta = hinge(x, y, c)
    return c, float(beta[1]), float(beta[1] + beta[2]), sse

def aic(sse, n, k):
    return n * np.log(sse / n) + 2 * k

def compare(x, y):
    _, _, sse1 = ols(x, y)
    c, lo, hi, sse2 = best_hinge(x, y)
    n = len(x)
    return c, lo, hi, aic(sse2, n, 4) - aic(sse1, n, 2)

def demean(v, g):
    out = np.empty_like(v, dtype=float)
    for p in np.unique(g):
        m = g == p
        out[m] = v[m] - v[m].mean()
    return out


## 2. Panel A — the form of the Shahzadi figure

Copy number against plasmid length, coloured by host chromosome length. Shahzadi report an exponent near **−0.71**. Section 4 below recovers −0.717 as the single power law on Maddamsetti's data, which is the dataset Shahzadi reuse — so that number is not independent evidence about this relationship.


In [ ]:
# @title Panel A
fig, ax = plt.subplots(figsize=(6.4, 4.6), dpi=120)
sc = ax.scatter(df.Logsize, df.y, c=df.Length_chr, s=6, alpha=0.5,
                cmap="coolwarm", linewidths=0,
                norm=Normalize(*np.quantile(df.Length_chr, [0.02, 0.98])))
fig.colorbar(sc, ax=ax).set_label("chromosome length (bp)")
k, b, _ = ols(df.Logsize.values, df.y.values)
xs = np.linspace(df.Logsize.min(), df.Logsize.max(), 100)
ax.plot(xs, k * xs + b, color="black", lw=1.6)
ax.set_xlabel("plasmid length, log10 bp")
ax.set_ylabel("copy number, log10")
ax.set_title(f"Ramiro subset: PCN $\\propto$ size$^{{{k:.2f}}}$ (n={len(df):,})", fontsize=11)
ax.axhline(0, color="#888888", lw=0.8)
plt.tight_layout(); plt.show()
print(f"this sample {k:+.3f}   Shahzadi report -0.71 on a different, larger sample")


## 3. Maddamsetti's own data

Supplementary Data 1 from [10.1038/s41467-025-61205-2](https://doi.org/10.1038/s41467-025-61205-2), fetched from the publisher's static host. PMC serves supplementary files behind a "Preparing to download" page that an automated request cannot pass; this host does not.

The file carries `AnnotationAccession` and `SeqID`, so unlike the Ramiro modelling subset it can be joined to external annotation.


In [ ]:
# @title Fetch Supplementary Data 1
mad_raw = fetch(MADDAM_URL, MADDAM_SHA, "Maddamsetti Supplementary Data 1")
md_all = pd.read_csv(io.BytesIO(mad_raw), encoding="utf-8", encoding_errors="replace")
mp = md_all[md_all.SeqType.str.strip().str.lower() == "plasmid"].copy()
mp = mp[(mp.PIRACopyNumber > 0) & (mp.normalized_replicon_length > 0)]
mp["x"] = np.log10(100 * mp.normalized_replicon_length)   # percent
mp["y"] = np.log10(mp.PIRACopyNumber)

print(f"{len(md_all):,} replicons -> {len(mp):,} usable plasmids")
print(f"columns: {', '.join(list(md_all.columns)[:8])} ...")
print(f"plasmids with copy number below 1: {int((mp.y < 0).sum()):,} "
      f"({100*(mp.y < 0).mean():.1f}%)  <- the Ramiro file has none")


## 4. Panel B — reproducing their figure

Their published bend is at **1.84%** of chromosome length with slopes **−0.880** and **−0.125**. This is what their file gives.


In [ ]:
# @title Panel B
c_m, lo_m, hi_m, d_m = compare(mp.x.values, mp.y.values)
k_m, b_m, _ = ols(mp.x.values, mp.y.values)

fig, ax = plt.subplots(figsize=(6.6, 4.6), dpi=120)
ax.scatter(mp.x, mp.y, s=4, alpha=0.35, linewidths=0, c="#56B4E9")
xs = np.linspace(mp.x.min(), mp.x.max(), 200)
_, beta = hinge(mp.x.values, mp.y.values, c_m)
ax.plot(xs, beta[0] + beta[1] * xs + beta[2] * np.maximum(xs - c_m, 0),
        color="#C1272D", lw=2)
ax.axhline(0, color="#888888", lw=0.8)
ax.set_xlabel("plasmid length as % of chromosome, log10")
ax.set_ylabel("copy number, log10")
ax.set_title(f"Maddamsetti Supplementary Data 1 (n={len(mp):,})", fontsize=11)
plt.tight_layout(); plt.show()

print(f"reproduced   bend {10**c_m:.2f}%  slopes {lo_m:+.3f} / {hi_m:+.3f}")
print(f"published    bend 1.84%      slopes -0.880 / -0.125")
print(f"single power law {k_m:+.3f}   (Shahzadi report -0.71)")


## 5. What our floor does to their data

`RESULTS.md` §3 notes that our lower slope is −0.683 where theirs is −0.880, and reasons that the PCN ≥ 1 floor cannot explain it because the floor barely touches small plasmids. That was an argument from the shape of our own data. Their file has the sub-1 plasmids ours lacks, so it can be tested directly.


In [ ]:
# @title Apply our floor to their data
rows = []
for lab, sub in (("their data, all plasmids", mp),
                 ("their data, PCN >= 1 (our floor)", mp[mp.PIRACopyNumber >= 1])):
    c, lo, hi, _ = compare(sub.x.values, sub.y.values)
    rows.append((lab, len(sub), 10**c, lo, hi))
c_r, lo_r, hi_r, _ = compare(df.x.values, df.y.values)
rows.append(("Ramiro subset (ours), PCN >= 1", len(df), 10**c_r, lo_r, hi_r))

print(f"{'subset':<36}{'n':>7}{'bend':>9}{'below':>9}{'above':>9}")
for lab, n, c, lo, hi in rows:
    print(f"{lab:<36}{n:>7}{c:>8.2f}%{lo:>9.3f}{hi:>9.3f}")
gap = rows[0][3] - rows[2][3]
closed = rows[0][3] - rows[1][3]
print(f"\nthe floor closes {100*closed/gap:.0f}% of the lower-slope gap,")
print("and moves the bend and the upper slope toward ours as well.")


## 6. The explorer

Three model modes, because two of them answer different questions and an earlier version of this notebook conflated them.

| mode | what it asks |
|---|---|
| **pooled** | is there a bend at all, ignoring family? |
| **PTU-adjusted, knee fixed** | does the bend *at 1.73%* survive giving each family its own intercept? |
| **PTU-centred, knee re-fitted** | is there a bend *anywhere* inside families once their means are gone? |

The middle one is the test that bears on the report's claim. The last one returns ≈ −1.3 and an earlier version of this notebook read that as the bend disappearing; it does not, because centring moves the 1.73% mark to a different coordinate for every family.


In [ ]:
# @title Explorer: three model modes
import ipywidgets as W
from IPython.display import display

MODES = ["pooled",
         "PTU-adjusted, knee fixed at 1.73%",
         "PTU-centred, knee re-fitted"]

def explore(mode, genus, mob, floor, min_ptu):
    d = df
    if genus != "(all)": d = d[d.Genus == genus]
    if mob != "(all)": d = d[d.predicted_mobility.astype(str) == mob]
    d = d[d.PCN >= floor]
    if min_ptu > 1:
        vc = d.Predicted_ptu.value_counts()
        d = d[d.Predicted_ptu.isin(vc[vc >= min_ptu].index)]
    if len(d) < 40:
        print(f"only {len(d)} plasmids left -- too few to fit"); return

    x, y, g = d.x.values, d.y.values, d.Predicted_ptu.values
    note = ""
    if mode == MODES[0]:
        c, lo, hi, dd = compare(x, y)
        _, beta = hinge(x, y, c)
        knee = c
    else:
        vc = d.Predicted_ptu.value_counts()
        d2 = d[d.Predicted_ptu.isin(vc[vc >= 3].index)]
        if len(d2) < 40:
            print("too few plasmids in families of 3 or more"); return
        g2 = d2.Predicted_ptu.values
        yd = demean(d2.y.values, g2); xd = demean(d2.x.values, g2)
        n = len(xd); npt = d2.Predicted_ptu.nunique()
        if mode == MODES[1]:
            knee = KNEE_PCT
            straddle = sum(1 for p in np.unique(g2)
                           if (d2.x.values[g2 == p] < knee).any()
                           and (d2.x.values[g2 == p] >= knee).any())
            if straddle == 0:
                print("no family straddles the knee -- not identified"); return
            zd = demean(np.maximum(d2.x.values - knee, 0.0), g2)
            M1 = xd[:, None]; M2 = np.column_stack([xd, zd])
            b1, *_ = np.linalg.lstsq(M1, yd, rcond=None)
            b2, *_ = np.linalg.lstsq(M2, yd, rcond=None)
            s1 = float(((yd - M1 @ b1) ** 2).sum())
            s2 = float(((yd - M2 @ b2) ** 2).sum())
            dd = aic(s2, n, npt + 3) - aic(s1, n, npt + 2)
            lo, hi = b2[0], b2[0] + b2[1]
            note = f"{straddle} of {npt} families straddle the knee"
            x, y = d2.x.values, d2.y.values
            beta = None
        else:
            x, y = xd, yd
            c, lo, hi, dd = compare(x, y)
            _, beta = hinge(x, y, c); knee = c
            note = f"centred on {npt} PTUs; the knee is in centred coordinates"

    fig, ax = plt.subplots(figsize=(6.6, 4.2), dpi=110)
    ax.scatter(x, y, s=5, alpha=0.4, linewidths=0, c="#56B4E9")
    xs = np.linspace(x.min(), x.max(), 200)
    if beta is not None:
        ax.plot(xs, beta[0] + beta[1] * xs + beta[2] * np.maximum(xs - knee, 0),
                color="#C1272D", lw=2)
    else:
        ax.axvline(knee, color="#C1272D", lw=1.4, ls="--")
        ax.text(knee, ax.get_ylim()[1], "  knee held here", va="top",
                fontsize=8, color="#C1272D")
    ax.set_xlabel("log10 % of chromosome" + (" (family-centred)" if mode == MODES[2] else ""))
    ax.set_ylabel("log10 copy number" + (" (family-centred)" if mode == MODES[2] else ""))
    ax.set_title(f"{mode}   n={len(x):,}   dAIC {dd:+.1f}", fontsize=10)
    plt.tight_layout(); plt.show()

    print(f"slopes {lo:+.3f} then {hi:+.3f}   change {hi-lo:+.3f}")
    print(f"dAIC   {dd:+.1f}")
    if note: print(note)

display(W.interactive(
    explore,
    mode=W.Dropdown(options=MODES, value=MODES[0], description="model"),
    genus=W.Dropdown(options=["(all)"] + sorted(df.Genus.dropna().unique().tolist()),
                     value="(all)", description="genus"),
    mob=W.Dropdown(options=["(all)"] + sorted(df.predicted_mobility.dropna().astype(str).unique().tolist()),
                   value="(all)", description="mobility"),
    floor=W.FloatSlider(value=1.0, min=1.0, max=10.0, step=0.5,
                        description="PCN floor", continuous_update=False),
    min_ptu=W.IntSlider(value=1, min=1, max=50, step=1,
                        description="min n per PTU", continuous_update=False)))


---

### What this notebook does not establish

- That the residual bend is biological. It survives PTU adjustment but these data cannot separate it from within-lineage curvature, PTU misclassification, the copy-number floor or sampling bias.
- That a PTU is a replication architecture. PTUs group plasmids by whole-sequence relatedness; shared PTU is not proof of shared replication control.
- That Shahzadi and Maddamsetti agree independently. Shahzadi reuse Maddamsetti's plasmids and copy-number estimates.
- Any uncertainty on a copy number. Neither source publishes one.
- That records are independent. Plasmids within a PTU are related by descent, so every AIC comparison here flatters itself.
